# E-Commerce Sales & Customer Analytics Project

Objective:
Analyze e-commerce transactional data using SQL to uncover insights related to revenue trends, customer behavior, product performance, and operational efficiency.

## Dataset Overview

Dataset contains:
- Orders
- Customers
- Products
- Payments
- Order Items

The project focuses on sales analytics, customer analytics, and business KPI generation.

In [ ]:
import pandas as pd
import sqlite3

In [ ]:
conn = sqlite3.connect('ecommerce.db')

In [ ]:
orders = pd.read_csv('olist_orders_dataset.csv')
products = pd.read_csv('olist_products_dataset.csv')
order_payments = pd.read_csv('olist_order_payments_dataset.csv')
order_items = pd.read_csv('olist_order_items_dataset.csv')
order_reviews = pd.read_csv('olist_order_reviews_dataset.csv')
customers = pd.read_csv('olist_customers_dataset.csv')

In [ ]:
orders.to_sql('orders', conn, if_exists='replace', index=False)
products.to_sql('products', conn, if_exists='replace', index=False)
order_payments.to_sql('order_payments', conn, if_exists='replace', index=False)
order_items.to_sql('order_items', conn, if_exists='replace', index=False)
order_reviews.to_sql('order_reviews', conn, if_exists='replace', index=False)
customers.to_sql('customers', conn, if_exists='replace', index=False)

Joins after observing Tables

In [ ]:
query="""
select  o.order_id, o.order_status, c.customer_city, c.customer_state
from    orders o
join    customers c on o.customer_id = c.customer_id
limit   10;
"""

pd.read_sql(query, conn)

In [ ]:
query="""
select  o.order_id, o.order_status, c.customer_state, p.payment_value
from    orders o
join    customers c on o.customer_id = c.customer_id
join    order_payments p on p.order_id = o.order_id
limit   10;
"""

pd.read_sql(query, conn)

KPI Queries

In [ ]:
# Total Orders
query="""
select  count(*) total_orders
from    orders;
"""

pd.read_sql(query, conn)

In [ ]:
# Total Customers
query="""
select  count(distinct(customer_id)) total_customers
from    customers;
"""

pd.read_sql(query, conn)

In [ ]:
# Total Revenue
query="""
select  round(sum(payment_value),2) total_revenue
from    order_payments;
"""

pd.read_sql(query, conn)

In [ ]:
# Orders By Status
query="""
select    order_status, count(order_status) total_orders
from      orders
group by  order_status
order by  total_orders desc;
"""

pd.read_sql(query, conn)

In [ ]:
# Top States By Orders
query="""
select    c.customer_state, count(customer_state) count_orders
from      customers c
join      orders o on c.customer_id = o.customer_id
group by  c.customer_state
order by  count_orders desc;
"""

pd.read_sql(query, conn)

In [ ]:
# Average Order Value
query="""
select    round(avg(payment_value), 3) avg_odr_value
from      order_payments;
"""

pd.read_sql(query, conn)

Time-Based Sales Analysis

In [ ]:
# Monthly Order Trend
query="""
select    strftime('%Y-%m', order_purchase_timestamp) order_month, count(order_id) total_orders
from      orders
group by  strftime('%Y-%m', order_purchase_timestamp)
order by  total_orders desc;
"""

pd.read_sql(query, conn)

In [ ]:
# Monthly Revenue Trend
query="""
select    strftime('%Y-%m', o.order_purchase_timestamp) order_month, round(sum(p.payment_value), 2) total_revenue
from      orders o
join      order_payments p on p.order_id = o.order_id
group by  strftime('%Y-%m', order_purchase_timestamp)
order by  total_revenue desc;
"""

pd.read_sql(query, conn)

In [ ]:
# Top 10 Products By Revenue
query="""
select    product_id, round(sum(price), 2) total_revenue
from      order_items
group by  product_id
order by  total_revenue desc
limit     10;
"""

pd.read_sql(query, conn)

In [ ]:
# Top Product Categories
query="""
select    p.product_category_name, round(sum(i.price), 2) total_revenue
from      order_items i
join      products p on p.product_id = i.product_id
group by  p.product_category_name
order by  total_revenue desc
limit     10;
"""

pd.read_sql(query, conn)

In [ ]:
# Average Delivery Time
query="""
select    round(avg(julianday(order_delivered_customer_date) - julianday(order_purchase_timestamp)), 2) avg_delivery_days
from      orders
where     order_delivered_customer_date is not null
"""

pd.read_sql(query, conn)

In [ ]:
# States With Highest Revenue
query="""
select    c.customer_state, round(sum(p.payment_value), 2) total_revenue
from      customers c
join      orders o on o.customer_id = c.customer_id
join      order_payments p on p.order_id = o.order_id
group by  c.customer_state
order by  round(sum(p.payment_value), 2) desc
limit     10;
"""

pd.read_sql(query, conn)

Advanced SQL Analytics

In [ ]:
# Top Customers By Revenue
query="""
select    c.customer_id, round(sum(p.payment_value), 2) total_revenue
from      customers c
join      orders o on o.customer_id = c.customer_id
join      order_payments p on p.order_id = o.order_id
group by  c.customer_id
order by  round(sum(p.payment_value), 2) desc
limit     10;
"""

pd.read_sql(query, conn)

In [ ]:
# Find Repeat Customers
query="""
select    c.customer_unique_id, count(o.customer_id) total_orders_placed
from      orders o
join      customers c on c.customer_id = o.customer_id
group by  c.customer_unique_id
having    count(o.customer_id) > 1
order by  count(o.customer_id) desc;
"""

pd.read_sql(query, conn)

In [ ]:
# State Ranking based on Revenue Using Window Function
query="""
select    customer_state,
          total_revenue,
          rank() over (order by total_revenue desc) state_rank
from
(
select    c.customer_state, round(sum(p.payment_value), 2) total_revenue
from      customers c
join      orders o on o.customer_id = c.customer_id
join      order_payments p on p.order_id = o.order_id
group by  c.customer_state
) state_data
"""

pd.read_sql(query, conn)

In [ ]:
# Running Revenue Total
query="""
select    order_month,
          total_revenue,
          sum(total_revenue) over (order by order_month desc) running_total
from
(
select    strftime('%Y-%m', o.order_purchase_timestamp) order_month, round(sum(p.payment_value), 2) total_revenue
from      orders o
join      order_payments p on p.order_id = o.order_id
group by  strftime('%Y-%m', o.order_purchase_timestamp)
) month_data;
"""

pd.read_sql(query, conn)

In [ ]:
# Top Product Categories Per State
query="""
select    customer_state,
          product_category_name,
          revenue
from
(
select    c.customer_state,
          p.product_category_name,
          round(sum(i.price), 2) revenue,
          rank() over (partition by c.customer_state order by sum(i.price) desc) rank_num
from      orders o
join      customers c on o.customer_id = c.customer_id
join      order_items i on i.order_id = o.order_id
join      products p on i.product_id = p.product_id
group by  c.customer_state, p.product_category_name
) state_productCategory_data
where     rank_num = 1
order by  customer_state;
"""

pd.read_sql(query, conn)

Data Cleaning + Professional SQL Structuring

SECTION 1 — NULL VALUE ANALYSIS

In [ ]:
# Missing Delivery Dates
query="""
select    count(*) missing_delivery_date
from      orders
where     order_delivered_customer_date is null;
"""

pd.read_sql(query, conn)

In [ ]:
# Missing Product Categories
query="""
select    count(*) missing_product_category
from      products
where     product_category_name is null;
"""

pd.read_sql(query, conn)

SECTION 2 — DUPLICATE CHECKS

In [ ]:
# Duplicate Orders Check
query="""
select    order_id, count(*) duplicate_orders
from      orders
group  by order_id
having    count(*)  > 1
order by  order_id;
"""

pd.read_sql(query, conn)

SECTION 3 — CASE WHEN

In [ ]:
# Customer Segmentation
query="""
select    customer_unique_id,
          total_payment,
          case
            when total_payment > 1000 then 'high_value'
            when total_payment > 500 then 'avg_value'
            else 'low_value'
          end as customer_segment
from
(
select    c.customer_unique_id, round(sum(p.payment_value), 2) total_payment
from      orders o
join      customers c on o.customer_id = c.customer_id
join      order_payments p on p.order_id = o.order_id
group  by c.customer_unique_id
) segment
order by  total_payment desc;
"""

pd.read_sql(query, conn)

SECTION 4 — CTEs

In [ ]:
# Revenue By State Using CTE
query="""
with state_revenue as
(
select    c.customer_state, round(sum(p.payment_value) ,2) revenue
from      orders o
join      customers c on o.customer_id = c.customer_id
join      order_payments p on o.order_id = p.order_id
group by  c.customer_state
)
select    *
from      state_revenue
order by  revenue desc;
"""

pd.read_sql(query, conn)

SECTION 5 — Multi-CTE Analytical Query

In [ ]:
# Revenue By Month using CTE and Rank
query="""
with
month_revenue as
(
select    strftime('%Y-%m', 	o.order_purchase_timestamp) order_month,
          round(sum(p.payment_value) ,2) revenue
from      orders o
join      order_payments p on o.order_id = p.order_id
group by  strftime('%Y-%m', 	o.order_purchase_timestamp)
),

ranked_month as
(
select    order_month, revenue,
          rank() over (order by revenue desc) ranked_revenue
from      month_revenue
)

select    *
from      ranked_month
order by  ranked_revenue;
"""

pd.read_sql(query, conn)

# Key Business Insights

1. Certain states contribute significantly higher revenue than others.

2. A small group of customers contributes disproportionately to sales.

3. Revenue shows clear monthly growth patterns.

4. Some product categories consistently outperform others.

5. Delivery delays may impact customer satisfaction and operational efficiency.

# Conclusion

This project demonstrates the use of SQL for business analytics, including KPI generation, customer analysis, revenue trend analysis, data cleaning, and advanced analytical querying using window functions and CTEs.